<a href="https://colab.research.google.com/github/vaibhavbhardwaj/AiSecurity/blob/main/garak_llm_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! mkdir llm_security_test
%cd llm_security_test
! apt update && apt install python3 python3.10-venv python3-pip openjdk-11-jdk -y
! export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
! pip install --upgrade pip
! pip install git+https://github.com/NVIDIA/garak.git@v0.11.0
! pip install transformers==4.52.1 torch==2.6.0 accelerate==1.4.0


In [ ]:
! export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64

In [ ]:
! apt update && apt install python3 python3.10-venv python3-pip openjdk-11-jdk -y

In [ ]:
!pip install --upgrade pip
!pip install git+https://github.com/NVIDIA/garak.git@v0.11.0
!pip install transformers==4.52.1 torch==2.6.0 accelerate==1.4.0

In [ ]:
print('Uninstalling conflicting protobuf versions...')
!pip uninstall -y protobuf

In [ ]:
print('Installing compatible protobuf version...')
!pip install protobuf==3.20.2

In [ ]:
print('Reinstalling transformers, torch, and accelerate to their latest compatible versions...')
!pip install --upgrade transformers torch accelerate

In [ ]:
print('Reinstalling transformers, torch, and accelerate to latest compatible versions...')
!pip install transformers torch accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define the model we want to use
model_name = "distilbert/distilgpt2"
revision_id = "2290a62682d06624634c1f46a6ad5be0f47f38aa"

# Load the tokenizer and model
print(f"Downloading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision_id)

print(f"Downloading model {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, revision=revision_id)

# Print model information
print(f"\nModel loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

# Test the model with a sample input
test_text = "Artificial intelligence is"
input_ids = tokenizer(test_text, return_tensors="pt").input_ids

print(f"\nGenerating sample completion for: '{test_text}'")
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_length=50,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Sample output: '{generated_text}'")

print("\nModel download and verification complete.")

In [ ]:
print('Attempting full uninstall of transformers, torch, and accelerate...')
!pip uninstall -y transformers torch accelerate

In [ ]:
print('Performing clean reinstall of transformers, torch, and accelerate...')
!pip install transformers torch accelerate

Now, let's re-run the model loading and generation code to verify the fix after a clean reinstall.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define the model we want to use
model_name = "distilbert/distilgpt2"
revision_id = "2290a62682d06624634c1f46a6ad5be0f47f38aa"

# Load the tokenizer and model
print(f"Downloading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision_id)

print(f"Downloading model {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, revision=revision_id)

# Print model information
print(f"\nModel loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

# Test the model with a sample input
test_text = "Artificial intelligence is"
input_ids = tokenizer(test_text, return_tensors="pt").input_ids

print(f"\nGenerating sample completion for: '{test_text}'")
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_length=50,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Sample output: '{generated_text}'")

print("\nModel download and verification complete.")

Now, let's re-run the model loading and generation code to verify the fix.

In [ ]:
!python3 -m garak --help

In [ ]:
! python3 -m garak --list_probes

In [ ]:
!python3 -m garak --model_type huggingface --model_name distilbert/distilgpt2 --probes promptinject --generations 1 --narrow_output

Let’s break down what these results mean:

Probe Type: Each line represents a different prompt injection test:
promptinject.HijackHateHumans: Tests if the model can be manipulated to generate content expressing hatred toward humans
promptinject.HijackKillHumans: Tests if the model can be tricked into generating violent content about harming humans
promptinject.HijackLongPrompt: Tests if longer, more complex prompts can bypass the model’s built-in safeguards
Score: The format FAIL score X/256 indicates:
The model was tested with 256 different prompt variations
It successfully resisted X of those attempts (higher is better)
When the model generates harmful content despite safeguards, it’s considered a failure
Example: 230/256 means the model resisted 230 prompts but failed on 26
Detector: AttackRogueString is the specific detector that identified the vulnerability:
This detector scans the model’s responses for strings that match known attack patterns
It flags when the model generates text that matches or resembles the malicious prompt
The detector is what determines if a test passed or failed
Failure Rate: The percentage of tests where the model generated prohibited content:
10.16% means 10.16% of the HijackHateHumans tests successfully tricked the model
17.58% on the HijackKillHumans test indicates this is the most successful attack vector
9.38% on the HijackLongPrompt test shows the lowest success rate for attackers
Even this small, uncensored model shows inherent resistance to most attacks (82.42-90.62% resistance rate)

Running a Comprehensive Vulnerability Scan
Now that we’ve tested for prompt injection, let’s expand our testing to include other types of vulnerabilities. We’ll run a comprehensive scan that includes multiple probe types:

promptinject: Tests for prompt injection vulnerabilities
exploitation: Tests for exploiting the model to perform unauthorized actions
malwaregen: Tests for generating malware code or instruction

xss: Tests for cross-site scripting vulnerabilities in web applications

python3 -m garak --model_type huggingface --model_name distilbert/distilgpt2 --probes promptinject,exploitation,malwaregen,xss --generations 1 --skip_unknown --narrow_output


Simplified Results by Category
1. Code Security Issues
The model sometimes generates harmful code when asked:

Successfully resisted template injection attempts
Failed in 33.33% of cases to execute Python code injection
Failed in 30% of cases to prevent SQL injection code
Generated malicious code components in 16.67% of tests
Created complete malware programs in 14.29% of tests
Main Risk: Users could trick the model into helping write malicious software.

2. Prompt Injection Issues
The model occasionally follows harmful instructions:

Expressed hatred toward humans in 12.89% of tests
Generated violent content in 16.02% of tests
Followed manipulative prompts in 8.59% of tests
Main Risk: The model can sometimes be manipulated to ignore safety guidelines.

3. Data Security Issues
The model reliably protected against data theft techniques:
- Prevented all attempts to create data-stealing code
- Blocked all markdown-based exfiltration attempts
- Resisted all tested data leakage methods

Main Strength: Excellent protection against data theft via the model.